# 03 — Gate 0 validation: does the stopping rule do what the protocol says?

Run **after** the arm solves from `02_solve.ipynb` (a0–a3, optional a4). Reads each arm's
`run_summary.json` + `portfolio_representation.csv` and renders the verdict tables for the
report-back. No solve happens here.

**G0 pass conditions** (spec Gate 0):
1. targeted carbon capture lands **at** its target (not above) — the target is a stopping rule,
   and `w = t` means it must not act as anything else;
2. the freed budget **visibly reallocates** — candidate destinations: climate corridors (0.96×
   area share in the control era) and the nine EFGs below area share;
3. *(a4, optional)* the pull-invariance arm reproduces the control **exactly** — the empirical
   proof that only `w/t` and the stopping point matter.

Fail branch (spec): revisit θ or the R2 test before any ensemble work — a conversation for the
chat, not a notebook edit.

**Kernel:** `Python (y2y-geo)`. Ethan runs; Claude never executes cells.

In [1]:
# ---- Setup + locate the arm runs ---------------------------------------------
import sys, pathlib
_cands = [p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
          if (p / "config.py").exists()]
assert _cands, f"config.py not found above {pathlib.Path.cwd()} -- run this notebook from inside the repo"
ROOT = _cands[0]
sys.path.insert(0, str(ROOT))

import importlib, json
import numpy as np
import pandas as pd

import config, leverage_core as lc, ensemble_core as ec
importlib.reload(config); importlib.reload(lc); importlib.reload(ec)

ARMS = ["a0_control", "a1_protocol", "a2_flat30", "a3_flat40", "a4_pullcheck"]  # a4 optional
# Generation: iter8 = binary MILP on Gurobi (the production formulation) when present,
# else fall back to the iter7 proportion-LP arms (the HiGHS relaxation record).
PREFIX = "iter8_y2y_" if any((config.RESULTS_DIR / f"iter8_y2y_{a}" / "run_summary.json").exists()
                             for a in ARMS) else "iter7_y2y_"
print(f"validating generation: {PREFIX}<arm>  "
      f"({'binary MILP / Gurobi' if PREFIX.startswith('iter8') else 'proportion LP / HiGHS'})")
RUNS = {}
for a in ARMS:
    d = config.RESULTS_DIR / f"{PREFIX}{a}"
    if (d / "run_summary.json").exists():
        RUNS[a] = d
    else:
        print(f"  (no solved run for {a} -- {'OPTIONAL, skipped' if a == 'a4_pullcheck' else 'REQUIRED, solve it in 02_solve'})")
assert all(a in RUNS for a in ARMS[:4]), "solve the four required arms in 02_solve before running this notebook"
print("arms found:", ", ".join(RUNS))

def rep(d):
    t = pd.read_csv(d / "portfolio_representation.csv").set_index("feature")["relative_held"]
    return t
def summ(d):
    return json.loads((d / "run_summary.json").read_text())
CAPTURE = {a: rep(d) for a, d in RUNS.items()}
SUMMARY = {a: summ(d) for a, d in RUNS.items()}

validating generation: iter8_y2y_<arm>  (binary MILP / Gurobi)
arms found: a0_control, a1_protocol, a2_flat30, a3_flat40, a4_pullcheck


## Verdict 1 — do the targets bind, and what does "bind" mean?

`w = t` on every arm, so a targeted pool's shortfall must reach zero. **Measured nuance (from the
superseded r1 run): min-shortfall never penalizes *exceeding* a target** — a satiated feature can
free-ride on cells selected for other values (r1: biomass landed at 0.259 against a 0.066 target,
purely incidentally, while m_soc parked at exactly 0.3320, its kink). So the verdicts are:

- **AT target (kink)** — capture ≈ target: the feature's residual cells weren't worth taking for
  anything else;
- **ABOVE target — incidental co-capture (expected)** — the excess is *free-riding*, reported as a
  number, not a failure;
- **BELOW target — investigate** — the only true failure: the stopping point wasn't reached.


In [2]:
# ---- capture vs target, per arm ------------------------------------------------
TOL = 0.005                     # LP tolerance on captured fraction
POOLS = ["irrecoverable_carbon_m_soc", "irrecoverable_carbon_biomass"]
rows = []
for a in RUNS:
    p = SUMMARY[a]["params"]
    tgts = p.get("targets") or {}
    for f in POOLS:
        t = float(tgts.get(f, 1.0))
        c = float(CAPTURE[a].get(f, np.nan))
        verdict = ("free (no target)"                 if t >= 0.999 else
                   "AT target (kink)"                 if abs(c - t) <= TOL else
                   f"ABOVE target -- incidental co-capture (+{c-t:.3f}, expected)" if c > t else
                   "BELOW target  <-- INVESTIGATE (stopping point not reached)")
        rows.append(dict(arm=a, feature=f.replace("irrecoverable_carbon_", ""),
                         target=round(t, 3), captured=round(c, 3),
                         delta=round(c - t, 4), verdict=verdict,
                         solve_s=round(SUMMARY[a]["solve_seconds"], 1)))
v1 = pd.DataFrame(rows)
print(v1.to_string(index=False))
bad = v1[v1.verdict.str.contains("INVESTIGATE")]
print("\nG0 condition 1:", "PASS -- every targeted pool reached its stopping point (excess above"
      " target, where present, is incidental co-capture and is REPORTED, not penalized)"
      if bad.empty else f"FAIL -- {len(bad)} pool(s) BELOW target; see above")

         arm feature  target  captured   delta          verdict  solve_s
  a0_control   m_soc   1.000     0.544 -0.4563 free (no target)     16.7
  a0_control biomass   1.000     0.413 -0.5867 free (no target)     16.7
 a1_protocol   m_soc   0.332     0.332  0.0000 AT target (kink)     81.2
 a1_protocol biomass   1.000     0.497 -0.5026 free (no target)     81.2
   a2_flat30   m_soc   0.300     0.300  0.0000 AT target (kink)   1196.5
   a2_flat30 biomass   0.300     0.300 -0.0000 AT target (kink)   1196.5
   a3_flat40   m_soc   0.400     0.400 -0.0000 AT target (kink)   1268.7
   a3_flat40 biomass   0.400     0.400  0.0000 AT target (kink)   1268.7
a4_pullcheck   m_soc   1.000     0.544 -0.4563 free (no target)     19.6
a4_pullcheck biomass   1.000     0.413 -0.5867 free (no target)     19.6

G0 condition 1: PASS -- every targeted pool reached its stopping point (excess above target, where present, is incidental co-capture and is REPORTED, not penalized)


## Verdict 2 — where did the freed budget go?

Full per-feature capture across arms, as deltas from the control. The prediction: the ~12–16
percentage points carbon gives up reappear in the under-served features — climate corridors and
the nine sub-area-share EFGs are the named candidates.

In [3]:
# ---- per-feature capture deltas vs control -------------------------------------
cont = lc.continuous_features()
efg_all = [f for f in CAPTURE["a0_control"].index if f not in cont
           and f != "irrecoverable_carbon_sl_soc"]
# the nine EFGs below area share in the CONTROL are the named candidates
efg_low = [f for f in efg_all if CAPTURE["a0_control"][f] < config.BUDGET_PCT]

tbl = pd.DataFrame({a: CAPTURE[a] for a in RUNS if a != "a4_pullcheck"})
out = tbl.loc[cont].copy()
out.loc["EFG mean (all)"] = tbl.loc[efg_all].mean()
out.loc[f"EFG mean (the {len(efg_low)} below area share)"] = tbl.loc[efg_low].mean()
for a in out.columns:
    if a != "a0_control":
        out[f"d {a}"] = out[a] - out["a0_control"]
print(out.round(3).to_string())
print(f"\nreading guide: positive deltas outside carbon = the reallocation the protocol predicts;")
print(f"climate_corridors sat at 0.96x area share and the {len(efg_low)} low EFGs at a control mean of "
      f"{tbl.loc[efg_low, 'a0_control'].mean():.3f}.")

                                   a0_control  a1_protocol  a2_flat30  a3_flat40  d a1_protocol  d a2_flat30  d a3_flat40
feature                                                                                                                  
human_modification                      0.304        0.302      0.299      0.301         -0.002       -0.005       -0.003
transboundary_connectivity              0.331        0.333      0.363      0.349          0.002        0.032        0.018
climate_corridors                       0.292        0.277      0.258      0.273         -0.014       -0.034       -0.018
climate_type_macrorefugia               0.388        0.399      0.431      0.414          0.011        0.043        0.026
irrecoverable_carbon_biomass            0.413        0.497      0.300      0.400          0.084       -0.113       -0.013
irrecoverable_carbon_m_soc              0.544        0.332      0.300      0.400         -0.212       -0.244       -0.144
aoh_richness_mammals    

## Verdict 3 — how much does the map move, and (optional) the pull-invariance proof

In [4]:
# ---- selected-set Jaccard vs control + the a4 pull-invariance check ---------
def sel(d):
    a = ec._alloc(d / "portfolio.tif")
    return (np.nan_to_num(a) > 0.5)
S0 = sel(RUNS["a0_control"])
print(f"{'arm':<14}{'Jaccard vs a0':>15}{'selected cells':>16}")
for a, d in RUNS.items():
    if a == "a0_control":
        continue
    S = sel(d)
    j = (S & S0).sum() / max((S | S0).sum(), 1)
    print(f"{a:<14}{j:15.4f}{int(S.sum()):16,}")

if "a4_pullcheck" in RUNS:
    S4 = sel(RUNS["a4_pullcheck"])
    same = bool((S4 == S0).all())
    j4 = (S4 & S0).sum() / max((S4 | S0).sum(), 1)
    if PREFIX.startswith("iter7"):
        print(f"\nG-uniform (a4, LP): identical cells -> {same}")
        print("   " + ("PASS -- w/t is the only pull parameter (exact reproduction)" if same else
                       "FAIL -- the w/t equivalence claim is WRONG somewhere; stop and report"))
    else:
        # Binary MILP with opt_gap g: the derivation guarantees a4 and a0 share the same ARGMIN
        # SET, but each solve may stop anywhere within g of optimum, and near-ties break
        # differently -- so the RIGHT equivalence test is the common OBJECTIVE VALUE, never
        # per-feature capture (a 1/40-weight EFG can swing its capture ~10 pts at negligible
        # objective cost; measured 2026-08-26). Reconstruct a0's objective from capture:
        cont = lc.continuous_features()
        efg = [f for f in CAPTURE["a0_control"].index
               if f not in cont and f != "irrecoverable_carbon_sl_soc"]
        obj = lambda h: (sum(1 - h[f] for f in cont)
                         + sum((1 - h[f]) / len(efg) for f in efg))
        o0, o4 = obj(CAPTURE["a0_control"]), obj(CAPTURE["a4_pullcheck"])
        rel = abs(o4 - o0) / o0
        g = float(SUMMARY["a4_pullcheck"]["params"].get("opt_gap", 0.001))
        band = 2.5 * g          # certificate is ~1-2x g; 2.5x allows reconstruction slop
        print(f"\nG-uniform (a4, binary): common-objective a0 {o0:.5f} vs a4 {o4:.5f} "
              f"(rel {rel:.2%}) | band {band:.2%} | Jaccard {j4:.4f} | "
              f"differing cells {int((S4 != S0).sum()):,}")
        if rel <= band:
            print("   PASS -- objective-equivalent; the differing cells at ~equal objective are a"
                  "\n   DEGENERACY datum for Gate 2 (interchangeable near-ties).")
        else:
            print("   INVESTIGATE -- difference exceeds the opt_gap certificate. Check the Gurobi"
                  "\n   logs for the ACHIEVED MIPGap of both solves, or re-solve one arm at"
                  "\n   opt_gap 1e-4 before reading this as a w/t violation. (The LP pair proved"
                  "\n   the equivalence exactly -- 0 differing cells -- so the claim itself stands;"
                  "\n   the question is only how loosely these two MILPs stopped.)")

arm             Jaccard vs a0  selected cells
a1_protocol            0.7784         381,874
a2_flat30              0.6293         381,874
a3_flat40              0.7917         381,874
a4_pullcheck           1.0000         381,874

G-uniform (a4, binary): common-objective a0 5.55784 vs a4 5.55784 (rel 0.00%) | band 0.03% | Jaccard 1.0000 | differing cells 0
   PASS -- objective-equivalent; the differing cells at ~equal objective are a
   DEGENERACY datum for Gate 2 (interchangeable near-ties).


## Report-back package

Everything the chat review needs: the frozen T2 (from notebook 01), F8, and the three verdict
tables above, plus `solve_seconds` per arm (the number that prices the future ensemble). **Stop
here** — Gate 1 (S0 construction), the manifest freeze, and all Gurobi work wait on that review.

## Relaxation tightness — LP (iter7) vs binary MILP (iter8)

Runs only when both generations exist. The measured answer to "did the LP relaxation mislead the
design phase?": per arm, the LP-vs-MILP selected-set Jaccard, the LP's own integrality, and the
largest per-feature capture difference. Supplementary-grade material for the methods section.

In [5]:
# ---- LP vs MILP tightness (needs both iter7_ and iter8_ generations) --------
pairs = [a for a in ARMS
         if (config.RESULTS_DIR / f"iter7_y2y_{a}" / "run_summary.json").exists()
         and (config.RESULTS_DIR / f"iter8_y2y_{a}" / "run_summary.json").exists()]
if not pairs:
    print("only one generation on disk -- nothing to compare")
for a in pairs:
    lp  = ec._alloc(config.RESULTS_DIR / f"iter7_y2y_{a}" / "portfolio.tif")
    mi  = ec._alloc(config.RESULTS_DIR / f"iter8_y2y_{a}" / "portfolio.tif")
    Slp, Smi = np.nan_to_num(lp) > 0.5, np.nan_to_num(mi) > 0.5
    frac = np.nan_to_num(lp)
    integral = float(((frac < 1e-6) | (frac > 1 - 1e-6) | ~np.isfinite(lp)).mean())
    j = (Slp & Smi).sum() / max((Slp | Smi).sum(), 1)
    c7 = pd.read_csv(config.RESULTS_DIR / f"iter7_y2y_{a}" / "portfolio_representation.csv").set_index("feature")["relative_held"]
    c8 = pd.read_csv(config.RESULTS_DIR / f"iter8_y2y_{a}" / "portfolio_representation.csv").set_index("feature")["relative_held"]
    print(f"  {a:<13} LP integrality {100*integral:6.2f}% | LP-vs-MILP Jaccard {j:.4f} | "
          f"max |capture delta| {float((c7 - c8).abs().max()):.4f}")

  a0_control    LP integrality 100.00% | LP-vs-MILP Jaccard 1.0000 | max |capture delta| 0.0000
  a1_protocol   LP integrality 100.00% | LP-vs-MILP Jaccard 0.9943 | max |capture delta| 0.0011
  a2_flat30     LP integrality 100.00% | LP-vs-MILP Jaccard 0.9997 | max |capture delta| 0.0004
  a3_flat40     LP integrality 100.00% | LP-vs-MILP Jaccard 0.9952 | max |capture delta| 0.0107
  a4_pullcheck  LP integrality 100.00% | LP-vs-MILP Jaccard 1.0000 | max |capture delta| 0.0000
